# 02. Baseline Ladder, Cost Policy Simulation & Held-Out Test Evaluation

**Course:** ES 335: Machine Learning (2026)  
**Protocol:** All tuning and threshold selection ($K^*$, $p^*$, $\tau^*$) are performed strictly via **5-fold `GroupKFold`** on the 80 development engines. The **20 held-out test engines** are evaluated once using locked development thresholds.

---

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT))

from src.data import load_config

cfg = load_config()
cv_reg_df = pd.read_csv(REPO_ROOT / 'results/metrics/cv_regression_ladder.csv')
cv_clf_df = pd.read_csv(REPO_ROOT / 'results/metrics/cv_classification_ladder.csv')
attr_df = pd.read_csv(REPO_ROOT / 'results/metrics/cv_feature_attribution.csv')

with open(REPO_ROOT / 'data/processed/test_evaluation_summary.json', 'r', encoding='utf-8') as f:
    test_summary = json.load(f)

print(f"Primary trailing window k = {cfg['features']['window_k']} cycles | Alarm horizon w = {cfg['framing']['alarm_w']} cycles")

## 1. 5-Fold `GroupKFold` Baseline Ladder (80 Dev Engines)
Comparing Rungs 0–5 for both continuous RUL regression (RMSE, MAE, NASA asymmetric score) and binary alarm classification (PR-AUC, Brier score, calibrated probability threshold $p^*$).

In [ ]:
display(cv_reg_df[['rung', 'model_name', 'rmse_all', 'mae_all', 'r2_all', 'nasa_asym_mean', 'nasa_asym_rul_le_50']])
display(cv_clf_df[['rung', 'model_name', 'pr_auc', 'brier_score', 'optimal_threshold_p', 'oof_policy_mean_cost_c100', 'failure_rate_pct']])

display(Image(filename=str(REPO_ROOT / 'results/figures/cv_baseline_ladder_progression.png')))
display(Image(filename=str(REPO_ROOT / 'results/figures/cv_pr_and_calibration_curves.png')))

### Feature Attribution (Lasso Sparsity vs. HistGradientBoosting Importance)
Lasso's L1 penalty drops redundant sensor features while keeping key thermal/pressure trends and engine age.

In [ ]:
print(f"Lasso retained {int(attr_df['lasso_selected'].sum())} / {len(attr_df)} features.")
display(attr_df.head(10))

## 2. Held-Out Test Evaluation & Cost Policy Simulation (20 Unseen Engines)
Applying the locked development thresholds ($K^*_{\text{dev}}$, $p^*_{\text{dev}}$, $\tau^*_{\text{dev}}$) once to the 20 held-out test engines across failure-to-waste cost ratios $c_{\text{fail}} : c_{\text{waste}}$, with 1,000-iteration engine-level bootstrap 95% confidence intervals.

In [ ]:
boot_df = pd.read_csv(REPO_ROOT / 'results/metrics/test_final_metrics.csv')
sweep_df = pd.read_csv(REPO_ROOT / 'results/metrics/cost_policy_comparison.csv')
official_df = pd.DataFrame(test_summary['official_fd001_100_engines_benchmark'])

print('Locked Dev Parameters (c_fail=100):', test_summary['locked_parameters_c100'])
print('Break-Even Cost Ratios:', test_summary['breakeven_analysis'])
display(boot_df[['model_or_comparison', 'metric_name', 'test_point_estimate', 'ci95_lower', 'ci95_upper', 'prob_positive_savings']])
display(sweep_df[['cost_ratio', 'fixed_K_dev', 'fixed_test_cost', 'champ_clf_test_cost', 'champ_clf_savings_vs_fixed_pct', 'tree_clf_test_cost', 'champ_reg_test_cost']])
display(Image(filename=str(REPO_ROOT / 'results/figures/cost_curve_vs_fixed_schedule.png')))

### Secondary Check: Official 100-Engine `test_FD001.txt` Benchmark
Sanity check at each truncated engine's final recorded cycle against `RUL_FD001.txt`.

In [ ]:
display(official_df[['rung', 'model_name', 'last_cycle_rmse_100_engines', 'last_cycle_mae_100_engines', 'last_cycle_r2_100_engines', 'last_cycle_nasa_sum_100_engines']])

## 3. Data Leakage Verification: Engine-Grouped Split vs. Random Row Split
Evaluating the same trailing-trend models (`Ridge`, `HistGradientBoosting`, `Random Forest`) under a naive random row split puts adjacent cycles ($t-1$, $t+1$) of the same engine into the training set, artificially deflating test RMSE.

In [ ]:
leakage_df = pd.read_csv(REPO_ROOT / 'results/metrics/leakage_experiment_results.csv')
display(leakage_df[['model_label', 'proper_groupkfold_cv_rmse', 'leaky_random_kfold_cv_rmse', 'proper_heldout_engines_test_rmse', 'leaky_random_row_split_test_rmse', 'artificial_error_reduction_pct']])
display(Image(filename=str(REPO_ROOT / 'results/figures/leakage_comparison_scatter.png')))

## 4. Sensitivity Analysis, Representative Error Audit & Limitations
Evaluating stability on 5-fold development CV across alarm horizon $w \in \{15, 30, 50\}$, RUL cap $\in \{125, 150, \text{Uncapped}\}$, trailing window $k \in \{10, 30\}$, and scaler isolation, alongside 3 representative held-out test engine error case studies.

In [ ]:
sens_df = pd.read_csv(REPO_ROOT / 'results/metrics/sensitivity_summary.csv')
display(sens_df[['sweep_dimension', 'setting_label', 'model_name', 'rmse_all', 'pr_auc', 'oof_policy_cost_c100', 'failure_rate_pct']])

cases = test_summary['error_case_studies']['case_studies']
case_rows = [
    {
        'role': cases[k]['label'],
        'unit_id': cases[k]['unit_id'],
        'T_fail': cases[k]['max_cycle'],
        'hgb_rmse': cases[k]['hgb_rmse'],
        'hgb_nasa_asym': cases[k]['hgb_nasa_asym_mean'],
        'fixed_cost': cases[k]['fixed_cost'],
        'logreg_cost': cases[k]['logreg_cost'],
        'hgb_cost': cases[k]['hgb_cost'],
    }
    for k in ('shortest_lived', 'longest_lived', 'worst_asymmetric_error')
]
display(pd.DataFrame(case_rows))
display(Image(filename=str(REPO_ROOT / 'results/figures/error_case_studies.png')))

### Key Scope & Limitations
- **Simulated data:** NASA C-MAPSS FD001 is simulator-generated with zero missing telemetry or unscheduled maintenance censoring.
- **Single operating regime:** FD001 spans one sea-level operating condition and one fault mode (HPC wear); cross-condition generalization (`FD002`–`FD004`) is out of scope.
- **Small test fleet:** Only 20 unseen test engines are available for full-lifetime policy simulation, causing wide bootstrap confidence intervals.
- **Assumed costs & non-causal attributions:** Failure-to-waste ratios ($10:1$ to $300:1$) represent illustrative maintenance trade-offs, and sensor attributions reflect correlation rather than root-cause physical intervention.